## Fine Tuning DistilBERT for Sentiment Analysis.

In [1]:
import pandas as pd 
import torch 
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import DistilBertModel, DistilBertTokenizerFast
import torch.optim as optim

In [2]:
df = pd.read_csv("twitter_multi_class_sentiment.csv")
df.shape

(16000, 3)

In [3]:
df.head()

,text,label,label_name
0,i didnt feel humiliated,0,sadness
1,i can go from feeling so hopeless to so damned...,0,sadness
2,im grabbing a minute to post i feel greedy wrong,3,anger
3,i am ever feeling nostalgic about the fireplac...,2,love
4,i am feeling grouchy,3,anger


In [4]:
from sklearn.preprocessing import LabelEncoder

In [5]:
label_encoder = LabelEncoder()
df['label_name'] = label_encoder.fit_transform(df['label_name'])

In [6]:
label_encoder.inverse_transform(df['label_name'])

array(['sadness', 'sadness', 'anger', ..., 'joy', 'anger', 'sadness'],
      dtype=object)

In [7]:
df.drop(columns = ['label'] , axis = 1 , inplace = True)

In [8]:
df.head()

,text,label_name
0,i didnt feel humiliated,4
1,i can go from feeling so hopeless to so damned...,4
2,im grabbing a minute to post i feel greedy wrong,0
3,i am ever feeling nostalgic about the fireplac...,3
4,i am feeling grouchy,0


In [9]:
import re 
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  # remove URLs
    text = re.sub(r"[^a-zA-Z0-9\s]", '', text)  # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [10]:
df['text'] = df['text'].apply(clean_text)

In [11]:
from sklearn.model_selection import train_test_split
train_data , test_data = train_test_split(
    df , test_size = 0.3, random_state = 42, stratify = df['label_name']
)
validation_data , test_data = train_test_split(
    test_data , test_size = 0.5, random_state = 42, stratify = test_data['label_name']
)

In [12]:
train_data.shape , test_data.shape , validation_data.shape

((11200, 2), (2400, 2), (2400, 2))

In [13]:
class SentimentDataset(Dataset): 
    def __init__(self , dataframe, tokenizer, max_length = 256): 
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return self.dataframe.shape[0]

    def __getitem__(self , index): 
        text = str(self.dataframe.iloc[index]['text'])
        label = self.dataframe.iloc[index]['label_name']

        # do the tokenization
        encoding = self.tokenizer(
            text, 
            padding = 'max_length', 
            max_length = self.max_length, 
            truncation = True, 
            return_tensors = 'pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item

In [14]:
train_data.iloc[1]['label_name']

2

In [15]:
model_checkpoint = 'distilbert-base-uncased'
distil_bert_tokenizer = DistilBertTokenizerFast.from_pretrained(
    model_checkpoint
)

In [16]:
temp = "Wow, that was great."
encodings = distil_bert_tokenizer(temp)

In [17]:
for key , val in encodings.items(): 
    val = torch.tensor(val)
    print(type(val))

<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [18]:
MAX_LEN = 256
train_dataset = SentimentDataset(
    dataframe = train_data, 
    tokenizer = distil_bert_tokenizer, 
    max_length = MAX_LEN
)
test_dataset = SentimentDataset(
    dataframe = test_data, 
    tokenizer = distil_bert_tokenizer, 
    max_length = MAX_LEN
)
validation_dataset = SentimentDataset(
    dataframe = validation_data, 
    tokenizer = distil_bert_tokenizer, 
    max_length = MAX_LEN
)

In [19]:
train_dataset[0]

{'input_ids': tensor([  101,  1045,  2514, 12511,  2009,  2003,  2053,  2393,  2005,  2033,
          2008,  2060,  5381,  2360,  2008,  1045,  2572,  3407,  2129,  2172,
          3606,  2045,  2089,  2022,  1999,  2009,   102,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,   

In [34]:
train_dataloader = DataLoader(
    train_dataset, batch_size = 16, shuffle = True
)
test_dataloader = DataLoader(
    test_dataset, batch_size = 16
)
validation_dataloader = DataLoader(
    validation_dataset, batch_size = 16
)

In [21]:
num_classes = len(label_encoder.classes_)

In [27]:
class DistilBERTSentimentClassifier(nn.Module): 
    def __init__(self , 
                 pretrained_model_name = 'distilbert-base-uncased', 
                 num_classes = 6, 
                 dropout = 0.3, 
                 device = None
                ): 
        
        super(DistilBERTSentimentClassifier , self).__init__()
        # load the base model 
        self.bert_model = DistilBertForSequenceClassification.from_pretrained(
               pretrained_model_name, num_labels = num_classes
        ).to(device)
        # freeze all the encoder block 
        for param in self.bert_model.parameters():
            param.requires_grad = False

        # Unfreeze the last encoder layer 
        for param in self.bert_model.distilbert.transformer.layer[-1].parameters(): 
            param.requires_grad = True 

    def forward(self , input_ids , attention_mask): 
        # pass these into bert model 
        outputs = self.bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        return outputs.logits

In [23]:
from transformers import DistilBertForSequenceClassification

In [25]:
bert_model = DistilBertForSequenceClassification.from_pretrained(
    model_checkpoint, num_labels = num_classes
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [26]:
bert_model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [28]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [29]:
model = DistilBERTSentimentClassifier(
    num_classes = num_classes, 
    device=device
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [30]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters() , lr = 2e-5)

In [31]:
model

DistilBERTSentimentClassifier(
  (bert_model): DistilBertForSequenceClassification(
    (distilbert): DistilBertModel(
      (embeddings): Embeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (transformer): Transformer(
        (layer): ModuleList(
          (0-5): 6 x TransformerBlock(
            (attention): DistilBertSdpaAttention(
              (dropout): Dropout(p=0.1, inplace=False)
              (q_lin): Linear(in_features=768, out_features=768, bias=True)
              (k_lin): Linear(in_features=768, out_features=768, bias=True)
              (v_lin): Linear(in_features=768, out_features=768, bias=True)
              (out_lin): Linear(in_features=768, out_features=768, bias=True)
            )
            (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affin

In [35]:
EPOCHS = 3
for epoch in range(EPOCHS): 
    model.train()
    total_loss = 0
    correct_train = 0 
    total_train = 0
    for batch in train_dataloader: 
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # forward pass 
        logits = model(input_ids , attention_mask)

        # loss 
        loss = criterion(logits , labels)
        # backward pass 
        loss.backward()
        # update grad 
        optimizer.step()
        total_loss += loss.item()
        # training acc 
        preds = torch.argmax(logits , dim = 1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    avg_train_loss = total_loss / len(train_dataloader)
    train_accuracy = correct_train / total_train
    

    # evaluate 
    model.eval()
    val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad(): 
        for batch in validation_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)

            loss = criterion(logits, labels)
            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)

    avg_val_loss = val_loss / len(validation_dataloader)
    val_accuracy = correct_val / total_val

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.4f}"
    )

Epoch [1/3] | Train Loss: 0.7466, Train Acc: 0.7621 | Val Loss: 0.6287, Val Acc: 0.8033
Epoch [2/3] | Train Loss: 0.6191, Train Acc: 0.8121 | Val Loss: 0.5534, Val Acc: 0.8296
Epoch [3/3] | Train Loss: 0.5470, Train Acc: 0.8391 | Val Loss: 0.5050, Val Acc: 0.8396


In [37]:
model.eval()
val_loss = 0
correct_val = 0
total_val = 0

with torch.no_grad(): 
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model(input_ids, attention_mask)

        loss = criterion(logits, labels)
        val_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct_val += (preds == labels).sum().item()
        total_val += labels.size(0)

avg_val_loss = val_loss / len(test_dataloader)
val_accuracy = correct_val / total_val

In [38]:
print(f"Test loss: {avg_val_loss} | Test Accuracy: {val_accuracy}")

Test loss: 0.46018497020006177 | Test Accuracy: 0.8616666666666667
